day05

0. 복습
데이터 스케일링
- 데이터의 특성들을 일정한 범위로 변화하는 과정

1) Min-Max 스케일링(정규화)
 : 모든 특성의 값을 0과 1사이의 범위로 변환

2) 표준화
 : 값을 평균 0, 표준편차 1

3) Robust 스케일링
 : 이상치에 강하다는 장점
   중앙값과 IQR(사분위 범위)을 기준으로 계산

데이터 프레임 결합

1) pd.concat()
- 그냥 이어 붙이는 가장 단순한 결합
- 이어붙일 데이터가 서로 같은 구조를 가져야 함

2) pd.merge()
- 공통 열(키) 값을 맞추기
- 서로 다른 정보를 가진 데이터를 키로 연결할때

1. 함수 매핑
 : 데이터의 여러 값에 같은 처리(함수)를 한꺼번에 적용
- 값 하나 하나에 똑같은 변환을 해야할때, for문으로 
  한칸 씩 도는 대신 한줄로 전부 처리한다
- map : 값을 대응표(딕셔너리 사용)로 바꾸거나 간단히 변환할때
	ex) male -> '남'
- apply : 값이나 행/열에 함수를 적용할때
	ex) 나이 -> "성인"/"미성년"

1) Series.map() : 값을 대응표로 바꾸기
- 한 열(Series)의 값들을 대응표(딕셔너리)에 따라 
  1:1로 치환, {원래값 : 바꿀값}
- sr.map({A:a, B:b})	딕셔너리로 치환
- sr.map(함수)		각 값에 함수 적용

2) Series.apply() : 각 값에 함수 적용
- apply는 직접 만든 함수를 각 값에 적용한다
- 조건에 따라 다른 결과를 내는 등 더 복잡한 변환에 쓴다

3) DataFrame.apply()
- 데이터 전체에 apply를 사용하면 
  함수가 열 또는 행 '한줄 통째로'를 받는다
- axis로 방향을 정한다
ex) axis=0(기본값) 열 별로 적용(함수가 열 하나를 통째로 받음)
    axis=1 : 행 별로 적용(함수가 행 하나를 통째로 받음)

4) DataFrame.map() : 모든 데이터에 하나씩 적용
- 데이터 프레임의 모든 원소에 같은 함수를 적용할때 사용
- apply가 열/행 '묶음'을 받은 것과 달리, 
  map은 원소 하나의 값을 받는다

5) 정리 
 대상		메서드		받는 값			용도
===========================================================================
 Series(한 열)	sr.map()	값 하나(혹은 딕셔너리)	값 치환
 Series(한 열)  sr.aplly()	값 하나			값 변환
 DataFrame	df.apply(axis=0/1)			열 요약,
				열/행 묶음		행 조합
 DataFrame	df.map()	값 하나			데이터에
							일괄 적용

2. 그룹 연산
 : 데이터를 어떤 기준으로 묶은 뒤, 묶음 별로 계산하는 것
ex) 각 반마다의 평균 점수, 서식지별 팽귄의 몸무게
- 그룹 연산은 3단계로 이뤄진다
	1단계 : 나누기(split) : 기준 열(ex:객실등급)의 값이 같은
		행끼리 묶는다(그룹화)
	2단계 : 적용(aplly) : 각 묶음(그룹)에 계산(평균, 합계 등)을
			      한다
	3단계 : 합치기(combine) : 그룹별 결과를 하나의 데이터로 모은다

1) groupby : 그룹별 집계하기
- df.groupby("기준열")["대상열"].집계함수() 형태로 사용
 : "기준열로 묶어서, 대상열을, 이렇게 계산해라"라는 뜻

2) 여러 기준으로 묶기
- 기준을 리스트로 여러개 작성하면, 그 조합별로 잘게 나눈다
ex) ['class', 'sex']로 묶으면 등급 x 성별 조합

3) 여러 통계를 한번에 - agg()
- groupby는 한번에 집계 하나만 구한다
- agg()를 사용하면 여러 통계를 한꺼번에 볼 수 있다
- 하나의 열에 여러 통계 - agg()에 집계 함수 이름들을 리스트로 넘긴다
- 열마다 다른 집계 - agg()에 딕셔너리를 주면, 열마다 다른 계산을 시킬 수 
		     있다. {열:집계함수}

In [ ]:
## 함수 매핑
import pandas as pd
import seaborn as sns

# 타이타닉 데이터 불러오기
titanic = sns.load_dataset("titanic")

# sibsp : 함께 탄 형제/배우자 수
# parch : 함께 탄 부모/자식 수

titanic.head()
### sr.map()
# {원래값:바꿀값} 딕셔너리 사용

# 성별 열의 male/female을 한글 남/여 로 한번에 바꾸기
gender_kor = titanic["sex"].map({"male" : "남", "female" : "여"})
gender_kor.head()
print(gender_kor.value_counts())
# 주의!
# 대응표(딕셔너리)에 없는 값은 NaN이 된다
# ex) 위 딕셔너리에 female을 빼먹으면, 모든 여성 값이 빈칸이 된다
# 그래서 map으로 치환할 때 가능한 값을 빠짐없이 작성해야 한다
###  sr.apply(함수)
# 나이를 받아 18세 이상은 '성인', 미만은 '미성년', NaN은 '나이모름'을 
# 반환하는 함수를 정의하여 적용

# 함수 정의
def age_group(age) :
    if age >= 18 :
        return "성인"
    elif age < 18 :
        return "미성년"
    else : 
        return "나이모름"

# age열의 각 값에 age_group()함수를 적용 후 새 열로 저장
titanic['연령구분'] = titanic['age'].apply(age_group)
titanic[['age', '연령구분']].head(8)
# Lambda로 짧게 작성

# 요금을 반올림해 정수로 

print(titanic['fare'].apply(lambda x : round(x)).head())
# 시리즈에서 map과 apply가 비슷하게 동작한다.
# 1:1 대응 치환 : map()
# 함수 적용(특히 복잡한 것)은 apply로 사용한다
## 데이터프레임 apply()
# axis = 0 : 각 '열'을 통째로 받아 계산
# => 열마다(최대값 - 최소값) 계산
nums = titanic[['age', 'fare']]
print(nums.apply(lambda col : col.max() - col.min()))
# col : 함수의 col은 열 하나(age 열, 그 다음 fare열)를 통째로 받는다
# 각 열의 값 범위가 나옴
# aixs=1 : 각 '행'을 통째로 받아 계산 -> 한 행에서 여러 열 조합
# 가족수 = 형제/배우자(sibsp) + 부모/자식(parch) + 본인 1명
titanic['가족수'] = titanic.apply(
    lambda row : row['sibsp'] + row['parch'] + 1, # row는 한사람(행) 전체
    axis=1 # 행 방향으로 적용
)

titanic[['sibsp', 'parch', '가족수']].head()
# age, fare열의 모든 데이터를 반올림
# 1) 결측치 제거 후 따로 저장
small = titanic[['age', 'fare']].dropna()
small.map(lambda x : round(x))
# 모든 데이터가 반올림됨
# <함수 매핑 실습>
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")
titanic.head()
# 1) class(객실등급 : First/Second/Third)를 map으로 한글
#    1등급/2등급/3등급
class_kor = titanic['class'].map({"First" : "1등급", "Second" : "2등급", "Third" : "3등급"})
print(class_kor.value_counts())
# 2) fare(요금) 50이상이면 "고가", 아니면 "저가"로 변환(aplly)
fare_label = titanic['fare'].apply(lambda x : "고가" if x >= 50 else "저가")
# 삼항 연산자 : 참일때 if 조건식 else 거짓일때
print(fare_label.head())

def fare_updown(fare) :
    if fare >= 50 :
        return "고가"
    else :
        return "저가"
# print(titanic['fare'].apply(fare_updown))
# 3) 행 단위(aplly(axis=1))로, sibsp + parch가 0이면 "혼자",
#    아니면 "동반"인 "동반여부"열 생성

titanic["동반여부"] = titanic.apply(
    lambda row : "혼자" if (row["sibsp"] + row["parch"]) == 0 else "동반",
    axis=1    
)
titanic[['sibsp', 'parch', "동반여부"]].head()
## 그룹 연산
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")
titanic.head()
# 객실 등급(class)으로 묶어 등급별 생존율(생존 여부의 평균)을 계산

# groupby('class') : 객실등급이 같은 사람끼리 묶는다
# ['survived'] : 그 중 생존여부 열을 대상으로
# .mean() : 묶음(그룹)마다 평균
titanic.groupby('class', observed=True)['survived'].mean()
# 등급별 '생존자 수' (survived 는 1/0이라 합계 = 생존자 수)
titanic.groupby('class', observed=True)['survived'].sum()
# 등급별 인원수
print(titanic['class'].value_counts())
print("==========================")
print(titanic.groupby('class', observed=True).size())
# 등급과 성별을 함께 묶어, "1등급 여성", "3등급 남성"처럼
# 조합별 생존률을 비교
titanic.groupby(['class', 'sex'], observed=True)['survived'].mean()
### agg()
# 등급별 나이의 평균, 최대, 개수를 한번에 확인
titanic.groupby('class', observed=True)['age'].agg(['mean', 'max', 'count'])
# 열마다 다른 집계
# 나이는 평균, 요금 최대, 생존여부는 합계(=생존자 수)를 등급별
titanic.groupby('class', observed=True).agg({
    "age":"mean", # 나이 -> 평균
    "fare":"max", # 요금 -> 최대값
    "survived":"sum" # 생존여부 -> 합계(생존자 수)
})
# +) groupby는 기준 열이 '인덱스(맨 왼쪽 이름)'가 된다
#    이걸 다시 보통 열로 되돌리면 일반적인 표 모양이 나온다
result = titanic.groupby('class', observed=True)['survived'].mean().reset_index()
result
# <그룹 연산 실습>
import pandas as pd
import seaborn as sns

tips = sns.load_dataset("tips")
tips.head()
# 1) 요일별(day) 평균 팁(tip)을 계산
tips.groupby('day', observed=True)['tip'].mean()
# 2) 흡연 여부(smoker)별 손님 그룹 수 확인
tips.groupby('smoker', observed=True)['size'].count()
# 3) 시간대(time)별 total_bill의 평균과 최대를 agg로 계산
tips.groupby('time', observed=True)['total_bill'].agg(['mean', 'max']).reset_index()

### 과제

## day05 과제

seaborn의 **`penguins`**(펭귄 344마리 측정 기록) 데이터로 연습합니다. 아래 셀을 먼저 실행하세요.

- 범주형 열 : `species`(종), `island`(섬), `sex`(성별)
- 수치형 열 : `bill_length_mm`(부리 길이), `bill_depth_mm`(부리 두께), `flipper_length_mm`(날개 길이), `body_mass_g`(몸무게)
- 앞부분(1~4번)은 `map`/`apply`로 값을 변환하고, 뒷부분(5~7번)은 `groupby`로 그룹별 집계를 합니다.

In [ ]:
import pandas as pd
import seaborn as sns

pg = sns.load_dataset("penguins")
print(pg.head())

문제 1) Series.map — 값을 대응표로 바꾸기

- `sex` 열의 `"Female"`/`"Male"` 을 한글 `"암컷"`/`"수컷"` 으로 한 번에 바꾸세요. (힌트 : `map({원래값: 바꿀값})`)
- 잘 바뀌었는지 `value_counts()`로 개수를 세어 출력하세요.

<출력결과>

sex
수컷    168
암컷    165
Name: count, dtype: int64

In [ ]:
pg['sex'].map({"Female" : "암컷", "Male" : "수컷"}).value_counts()

문제 2) Series.apply — 함수로 값 변환하기

- 몸무게(`body_mass_g`) 하나를 받아 아래 규칙대로 문자열을 돌려주는 함수를 만들어 `apply`로 적용하고, 새 열 `몸무게구분` 에 저장하세요.
    - 4500 이상 → `"무거움"`, 3500 이상 → `"보통"`, 3500 미만 → `"가벼움"`, 그 외(빈칸 `NaN`) → `"모름"`
- `body_mass_g`와 `몸무게구분`을 앞 6줄 출력하고, `몸무게구분`의 개수도 세어 출력하세요.

<출력결과>

body_mass_g 몸무게구분
0       3750.0    보통
1       3800.0    보통
2       3250.0   가벼움
3          NaN    모름
4       3450.0   가벼움
5       3650.0    보통
몸무게구분
보통     153
무거움    118
가벼움     71
모름       2
Name: count, dtype: int64

In [ ]:
def body_kr(body) :
    if body >= 4500 :
        return "무거움"
    elif body >= 3500 :
        return "보통" 
    elif body < 3500 :
        return "가벼움"
    else :
        return "모름"

pg["몸무게구분"] = pg["body_mass_g"].apply(body_kr)
print(pg[['body_mass_g', "몸무게구분"]].head(6))
print(pg["몸무게구분"].value_counts())

문제 3) DataFrame.apply(axis=1) — 여러 열을 조합해 새 열 만들기

- 행 단위(`axis=1`)로, **부리 길이 ÷ 부리 두께**(`bill_length_mm / bill_depth_mm`)를 소수 둘째 자리까지 계산해 새 열 `부리비율` 에 저장하세요. (힌트 : `apply(lambda row: ..., axis=1)`)
- `bill_length_mm`, `bill_depth_mm`, `부리비율` 을 앞 5줄 출력하세요. (값이 빈 3번 행은 결과도 `NaN`)

<출력결과>

bill_length_mm  bill_depth_mm  부리비율
0            39.1           18.7  2.09
1            39.5           17.4  2.27
2            40.3           18.0  2.24
3             NaN            NaN   NaN
4            36.7           19.3  1.90

In [ ]:
pg["부리비율"] = pg.apply(lambda row : round(row['bill_length_mm'] / row['bill_depth_mm'], 2), axis=1)
pg[['bill_length_mm', "bill_depth_mm", '부리비율']].head()

문제 4) DataFrame.map — 모든 칸에 하나씩 적용

- `bill_length_mm`, `body_mass_g` 두 열의 **앞 3줄**만 뽑아, 표의 **모든 칸을 반올림**한 결과를 출력하세요. (힌트 : `df.map(lambda x: round(x))`)

<출력결과>

bill_length_mm  body_mass_g
0              39         3750
1              40         3800
2              40         3250

In [ ]:
pg[['bill_length_mm', "body_mass_g"]].dropna().map(lambda x : round(x)).head(3)

문제 5) groupby 기본 — 그룹별로 집계하기

- **종(`species`)별 평균 몸무게(`body_mass_g`)**를 소수 첫째 자리까지 구해 출력하세요. (힌트 : `groupby("species")["body_mass_g"].mean()`)
- **섬(`island`)별 펭귄 마리 수**를 `size()`로 세어 출력하세요.

<출력결과>

species
Adelie       3700.7
Chinstrap    3733.1
Gentoo       5076.0
Name: body_mass_g, dtype: float64
island
Biscoe       168
Dream        124
Torgersen     52
dtype: int64

In [ ]:
print(pg.groupby('species')['body_mass_g'].mean().round(1))
print(pg.groupby('island').size())

문제 6) agg — 여러 통계를 한 번에

- **종(`species`)별** `body_mass_g`의 **평균·최대·개수**를 `agg`로 한 번에 구해 소수 첫째 자리까지 출력하세요. (힌트 : `agg(["mean", "max", "count"])`)

<출력결과>

mean     max  count
species
Adelie     3700.7  4775.0    151
Chinstrap  3733.1  4800.0     68
Gentoo     5076.0  6300.0    123

In [ ]:
pg.groupby('species')['body_mass_g'].agg(["mean", "max", "count"]).round(1)

문제 7) 여러 기준으로 묶기 + reset_index

- **종(`species`) × 성별(`sex`)** 조합별 **평균 몸무게**를 소수 첫째 자리까지 구하세요. (힌트 : `groupby(["species", "sex"])`)
- 결과에 `reset_index()`를 붙여 **깔끔한 표**로 정리해 출력하세요.

<출력결과>

species     sex  body_mass_g
0     Adelie  Female       3368.8
1     Adelie    Male       4043.5
2  Chinstrap  Female       3527.2
3  Chinstrap    Male       3939.0
4     Gentoo  Female       4679.7
5     Gentoo    Male       5484.8

In [ ]:
print(pg.groupby(["species", "sex"])['body_mass_g'].mean().reset_index())